In [1]:
import pandas as pd
import numpy as np
import os

from tqdm import tqdm
import ast
import re

import faiss
from uuid import uuid4

from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

# ======= config path =======
BASE_DIR = '/project/lt200304-dipmt/paweekorn'
MODEL_PATH = f"{BASE_DIR}/models/retriever/bge-m3"
RESULT_DIR = f"{BASE_DIR}/vectorstore/{os.path.basename(MODEL_PATH)}"

## Overview

In [2]:
def clean_parenthesis(text):
    text = text.replace('[', '(')
    text = text.replace(']', ')')
    return text

def clean_thai_spacing(text):
    filtered = re.findall(r'[^\u0E00-\u0E7F]', text)
    if all([x.isspace() for x in filtered]):
        text = re.sub(r'\s', '', text)
    return text

In [5]:
test_df = pd.read_csv(f"{BASE_DIR}/data/DS01/test_v1.csv")

unique_df = pd.read_csv(f"{BASE_DIR}/data/unique_no_test.csv")
unique_df['ENG'] = unique_df['ENG'].apply(clean_parenthesis)
unique_df['THA'] = unique_df['THA'].apply(clean_parenthesis)
unique_df['NAME'] = unique_df['NAME'].apply(lambda x: ast.literal_eval(x)[0])

unique_df = unique_df.drop_duplicates('ENG')
unique_df = unique_df[ unique_df['ENG'].apply(lambda x: x not in test_df['ENG'].tolist()) ]

print(unique_df.shape)
unique_df.head()

(191751, 3)


,ENG,THA,NAME
0,"(Animal) skin, pelt","(สัตว์) หนัง, ขนสัตว์",18
1,(IaaS) infrastructure a a service,ให้บริการโครงสร้างพื้นฐานด้านไอที (ไอเอเอเอส),42
2,(abrasive preparation) soap,(สารที่เตรียมขึ้นใช้ขัด) สบู่,3
3,all good of textile,สินค้าทั้งหมดที่ทำจากสิ่งทอ,24
4,(audio-video) disc,(เสียง-วีดีโอ) แผ่นดิสก์,9


In [4]:
documents = []
for _, row in tqdm(unique_df.iterrows()):
    content = row['ENG']
    meta = {'thai': row['THA'], 'wipo': row['NAME']}
    documents.append(Document(page_content=content, metadata=meta))

uuids = [str(uuid4()) for _ in range(len(documents))]
documents[0]

191713it [00:06, 28794.53it/s]


Document(metadata={'thai': '(สัตว์) หนัง, ขนสัตว์', 'wipo': '18'}, page_content='(Animal) skin, pelt')

## vectorstore

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name=MODEL_PATH)
d_model = len(embedding_model.embed_query('Hello World'))
d_model

1024

In [6]:
res = faiss.StandardGpuResources()

index_flat = faiss.IndexFlatL2(d_model)
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)

# make it into a gpu index
vector_store = FAISS(
    embedding_function=embedding_model,
    index=gpu_index_flat,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
doc_ids = vector_store.add_documents(documents=documents, ids=uuids)
print(doc_ids[:5])

['4498b18c-9068-4b25-a588-37b307cb09f8', 'c5b1b0fa-f570-4e65-8abe-9df0f0445f7e', 'd860e740-b701-4c0d-9e77-95ebc9c94ba8', 'ba6a94ce-965f-4516-96a5-b4478d47695c', '6979debc-87b3-4b9a-b64a-4a22f14e725d']


In [ ]:
# Move the index to CPU before saving
os.makedirs(f"{RESULT_DIR}", exist_ok=True)
cpu_index = faiss.index_gpu_to_cpu(vector_store.index)
faiss.write_index(cpu_index, f"{RESULT_DIR}/index.faiss")

# Re-create the FAISS vector store from the saved index, docstore, and index_to_docstore_id
vector_store_loaded = FAISS(
    embedding_function=embedding_model,
    index=cpu_index,
    docstore=vector_store.docstore,
    index_to_docstore_id=vector_store.index_to_docstore_id,
)

# Now you can save the other components using save_local
vector_store_loaded.save_local(RESULT_DIR)

## Demo

In [12]:
def get_relevant_docs(query, k=3):
    docs = vector_store.similarity_search(query, k=k)

    relevant = ""
    for i, doc in enumerate(docs[1:]):
        relevant += f'''English: {doc.page_content}
Thai: {doc.metadata['thai']}
\n'''

    return relevant

sample = "socket, plug and other contact (electric connection)"
print(f"Source: {sample}\n")

print("## Retrieved References")
print(get_relevant_docs(sample))

Source: socket, plug and other contact (electric connection)

## Retrieved References
English: plug, socket and other contact (electrical connection)
Thai: ปลั๊ก, เต้ารับ และ หน้าสัมผัสอื่นๆ (การเชื่อมต่อไฟฟ้า)

English: Plugs, socket and other contact (electric connection)
Thai: ปลั๊ก, เต้ารับ และอุปกรณ์เชื่อมต่อไฟฟ้า (การเชื่อมต่อทางไฟฟ้า)




## Database for full-text search

In [7]:
import sqlite3

local_db = sqlite3.connect("unique.db")
cur_local = local_db.cursor()

# Create a table to store the data
cur_local.execute('DROP TABLE IF EXISTS unique_data;')
cur_local.execute('CREATE TABLE unique_data (ENG TEXT, THA TEXT, NAME INTEGER);')

unique_df.to_sql('unique_data', local_db, if_exists='replace', index=False)

local_db.commit()
local_db.close()